In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('messy_ecommerce_sales_data.csv')
df.head()


,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


In [2]:
# cek duplikasi dan missing value
duplicate_data = df.duplicated().sum()
print(f"Jumlah data duplikat: {duplicate_data}")

missing_data = df.isnull().sum()
print(f"Jumlah data missing: {missing_data}")

Jumlah data duplikat: 1
Jumlah data missing: ID                 0
 Customer_Name     0
Order_ID           0
Order_Date         0
Product            0
 Category          8
Quantity           5
Price              5
Payment_Method     0
Status             0
Total             14
dtype: int64


In [3]:
# hilangkan duplikat
df.drop_duplicates(inplace=True)
cek_duplikacates = df.duplicated().sum()
print(f"Jumlah data duplikat: {cek_duplikacates}")


Jumlah data duplikat: 0


In [ ]:
# cek type data
print(df.dtypes)

# Numeric columns
print("\n=== NUMERIC SUMMARY ===")
print(df.describe())

# Unique values
print("\n=== UNIQUE VALUES ===")
for col in df.columns:
    print(f"\n{col}:")
    print(df[col].unique())

ID                  int64
 Customer_Name     object
Order_ID           object
Order_Date         object
Product            object
 Category          object
Quantity           object
Price              object
Payment_Method     object
Status             object
Total             float64
dtype: object

=== NUMERIC SUMMARY ===
               ID         Total
count  102.000000     88.000000
mean   149.676471   1230.927614
std     28.843779   2620.332451
min    100.000000 -20000.000000
25%    125.250000    574.725000
50%    149.500000   1127.180000
75%    174.750000   2179.652500
max    199.000000   4722.700000

=== UNIQUE VALUES ===

ID:
[100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117
 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135
 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153
 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171
 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 18

In [ ]:
# Cek tabel nya
# Quantity harus numeric
quantity = pd.to_numeric(df['Quantity'], errors='coerce')

invalid_quantity = df[
    df['Quantity'].notna() & quantity.isna()
]

# Quantity tidak boleh negatif
negative_quantity = df[
    quantity < 0
]

# Price harus numeric
price = pd.to_numeric(df['Price'], errors='coerce')

invalid_price = df[
    df['Price'].notna() & price.isna()
]

# Price tidak boleh negatif
negative_price = df[
    price < 0
]

# Total tidak boleh negatif
negative_total = df[
    df['Total'] < 0
]

print("Invalid Quantity:")
print(invalid_quantity)

print("\nNegative Quantity:")
print(negative_quantity)

print("\nInvalid Price:")
print(invalid_price)

print("\nNegative Price:")
print(negative_price)

print("\nNegative Total:")
print(negative_total)

Invalid Quantity:
     ID  Customer_Name   Order_ID Order_Date Product  Category Quantity  \
92  192   Customer_192  ORD-35144        abc  Jacket  Clothing       4a   

     Price Payment_Method    Status  Total  
92  203.63    Credit Card  Returned    NaN  

Negative Quantity:
     ID  Customer_Name   Order_ID Order_Date  Product    Category Quantity  \
17  117   Customer_117  ORD-72751  2/12/2025  Blender  electronic       -2   
34  134   Customer_134  ORD-16585  10/4/2025  T-shirt    Clothing       -5   

     Price    Payment_Method      Status     Total  
17   10000  Cash on Delivery  Processing -20000.00  
34  591.53            PayPal     Shipped  -2957.65  

Invalid Price:
     ID  Customer_Name   Order_ID  Order_Date     Product     Category  \
1   101   Customer_101  ORD-35783    7/5/2025  Smartphone  Electronics   
10  110   Customer_110  ORD-61020   9/26/2025  Basketball       sports   
20  120   Customer_120  ORD-14659   6/13/2025    Yoga Mat  ELECTRONICS   
96  196   Custo

In [11]:
# Rapikan Nama Kolom (Hapus spasi liar & ubah ke huruf kecil)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Bersihkan Kolom Harga / Numerik (Hapus huruf, Rp, $, dan koma)
numeric_columns = [
    'price', 
    'unit_price',
    'total_price',
    'amount',
    'sales',
    'quantity'
]

for col in numeric_columns:
    if col in df.columns:
        # Konversi ke string dulu, lalu hapus karakter selain angka dan titik
        df[col] = (
            df[col].astype(str).str.replace(r"[^\d.]", "", regex=True)
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Standarisasi Kolom Tanggal
date_cols = ['date', 'order_date', 'transaction_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce') # Tanggal yang rusak diubah jadi NaT

# Bersihkan Kolom Teks (Hapus spasi & samakan kapitalisasi)
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace(["Nan", "None", ""], np.nan)

# Filter Nilai Negatif / Tidak Valid
if 'quantity' in df.columns:
    df = df[df['quantity'] > 0]

if 'price' in df.columns:
    df = df[df['price'] > 0]

# Hapus Duplikat
df = df.drop_duplicates()

print("\n=== Tampilan Data Setelah Dibersihkan ===")
print(df.info())
print(df.head())



=== Tampilan Data Setelah Dibersihkan ===
<class 'pandas.core.frame.DataFrame'>
Index: 90 entries, 0 to 101
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              90 non-null     int64         
 1   customer_name   90 non-null     object        
 2   order_id        90 non-null     object        
 3   order_date      88 non-null     datetime64[ns]
 4   product         90 non-null     object        
 5   category        83 non-null     object        
 6   quantity        90 non-null     float64       
 7   price           90 non-null     float64       
 8   payment_method  90 non-null     object        
 9   status          90 non-null     object        
 10  total           88 non-null     float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(6)
memory usage: 8.4+ KB
None
    id customer_name   order_id order_date        product category  quantity  \
0  100  Customer

In [12]:
df.to_csv('cleaned_ecommerce_sales_data.csv', index=False)
print("\nData bersih berhasil disimpan ke 'cleaned_ecommerce_sales_data.csv'!")


Data bersih berhasil disimpan ke 'cleaned_ecommerce_sales_data.csv'!
